# Chạy thí nghiệm chính thức trên Colab (A100)

Notebook này dùng cho **các lần đo chính thức** (thời gian, kết quả cuối) trên tài khoản Colab của Huấn, để mọi kết quả trong báo cáo chạy trên cùng một phần cứng.

Thứ tự chạy:

1. Chọn runtime GPU (Runtime > Change runtime type > A100).
2. Sửa `REPO_URL` và `REF` ở ô cấu hình, rồi chạy lần lượt từng ô.
3. Kết quả được lưu vào Google Drive sau mỗi thí nghiệm, nên nếu Colab ngắt kết nối thì chỉ cần chạy lại: các lần chạy đã có trong CSV sẽ được bỏ qua (resume).
4. Cuối cùng tải `results.zip` về và commit các file CSV vào nhánh của mình.

In [ ]:
# Cấu hình
REPO_URL = "https://github.com/<owner>/tabular-dl-vs-gbdt.git"  # TODO: sửa khi nhóm tạo repo
REF = "main"              # nhánh hoặc tag, ví dụ "v1.0"
USE_DRIVE = True          # lưu kết quả vào Google Drive để không mất khi ngắt kết nối
DRIVE_DIR = "/content/drive/MyDrive/CO5175_topic3_results"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Không có GPU"

In [ ]:
import os, subprocess
if not os.path.exists("/content/repo"):
    subprocess.run(["git", "clone", REPO_URL, "/content/repo"], check=True)
%cd /content/repo
!git fetch --all --tags -q && git checkout -q {REF} && git pull -q || true
!git log -1 --oneline

In [ ]:
!pip install -q -r requirements.txt
!mkdir -p results/env && pip freeze > results/env/requirements-lock.txt
!python --version | tee results/env/python_version.txt

In [ ]:
# Nối thư mục results với Google Drive (giữ kết quả khi Colab ngắt kết nối)
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    !cp -rn results/. {DRIVE_DIR}/ 2>/dev/null || true
    !rm -rf results && ln -s {DRIVE_DIR} results
!ls -la results/

## Kiểm tra nhanh trước khi chạy thật

Chạy test và smoke test (dữ liệu giả, vài giây). Nếu ô này lỗi thì dừng lại, đừng chạy thí nghiệm.

In [ ]:
!pytest -q
!python scripts/run_baseline.py --smoke

## Thí nghiệm chính thức

Mỗi ô ghi vào một file CSV trong `results/`. Thêm `--models` hoặc `--datasets` để chạy một phần, ví dụ `--models ft_transformer --datasets adult`.

In [ ]:
!python scripts/run_baseline.py           # mục 3.2

In [ ]:
!python scripts/run_full.py               # mục 3.3, 3.4, 3.8 (tuning + 10 seed)

In [ ]:
!python scripts/run_ablation_budget.py    # mục 3.5, Hình 2

In [ ]:
!python scripts/run_size_curve.py         # mục 3.6, Hình 3

In [ ]:
!python scripts/run_tabpfn.py             # mục 3.7 (bỏ qua nếu nhóm bỏ TabPFN)

## Bảng, hình và tải kết quả

In [ ]:
!python scripts/make_report.py
!cat results/tables/table2_main.md

In [ ]:
!cd /content && rm -f results.zip && zip -qr results.zip repo/results/ && ls -lh results.zip
from google.colab import files
files.download("/content/results.zip")